# Cybersecurity Guidelines Assistant — RAG Pipeline

**Domain:** official NIST cybersecurity publications (framework, incident response, authentication, risk assessment, patching, malware, media sanitization, contingency planning, small-business security).

**Pipeline:** PDFs → clean → chunk → embed (sentence-transformers) → Chroma (persisted) → retrieve → grounded prompt → local Ollama LLM → cited answer.

**Track:** Core (text-only).

**How to run:** start Ollama (`ollama pull llama3.2:3b`), put the PDFs in `data/raw_pdfs/` (see the root README), then use *Kernel → Restart & Run All*.

In [ ]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import chromadb
import ollama
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Paths (works whether the notebook is started from the repo root or notebooks/) ----
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw_pdfs"
VECTOR_DIR = ROOT / "backend" / "data" / "vector_store"   # <- the backend loads this folder
CHROMA_DIR = VECTOR_DIR / "chroma"
DOCS_DIR = ROOT / "docs"

# ---- Settings (all exported to config.json in section 2.7) ----
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 1000        # characters
CHUNK_OVERLAP = 200      # characters
MIN_CHUNK_CHARS = 150    # drop tiny fragments
COLLECTION_NAME = "nist_docs"
TOP_K = 4
MAX_DISTANCE = 0.65      # cosine distance; tuned in section 2.6
OLLAMA_MODEL = "llama3.2:3b"
TEMPERATURE = 0.1

pdf_files = sorted(RAW_DIR.glob("*.pdf"))
print("Project root:", ROOT)
print(f"Found {len(pdf_files)} PDF(s) in {RAW_DIR}")
assert pdf_files, "No PDFs found - download them first (see README) and put them in data/raw_pdfs/"

## 2.1 Load & Inspect

Every PDF is read page by page with `pypdf`. For each page we keep the **filename** and **page number** so every chunk can later be cited.

In [ ]:
def load_pdf(path: Path):
    """Return (pages, error). pages = list of {source, page, text}."""
    try:
        reader = PdfReader(str(path))
        if reader.is_encrypted:
            reader.decrypt("")
        pages = []
        for i, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception:
                text = ""
            pages.append({"source": path.name, "page": i, "text": text})
        return pages, None
    except Exception as exc:
        return [], str(exc)


raw_pages, inspect_rows = [], []
for f in pdf_files:
    pages, error = load_pdf(f)
    raw_pages.extend(pages)
    n = len(pages)
    total_chars = sum(len(p["text"]) for p in pages)
    empty = sum(1 for p in pages if len(p["text"].strip()) < 50)
    avg = total_chars / n if n else 0
    if error:
        status = f"FAILED: {error}"
    elif avg < 200:
        status = "needs OCR? (very little text)"
    else:
        status = "ok"
    inspect_rows.append({"file": f.name, "pages": n, "chars": total_chars,
                         "avg_chars_per_page": round(avg), "near_empty_pages": empty, "status": status})

inspect_df = pd.DataFrame(inspect_rows)
display(inspect_df)
print(f"\nDocuments: {len(pdf_files)} | Pages: {len(raw_pages)} | "
      f"Failed/needs OCR: {(inspect_df['status'] != 'ok').sum()}")

**Findings (edit after running):**

- **Documents / pages:** *N documents, ~N pages in total.*
- **Formats:** all PDF (text-extractable).
- **Failed to parse / need OCR:** *none* (or list the files flagged above and what you did — replaced or removed them).
- **Messiness noticed:** repeated headers/footers and page numbers, table-of-contents dot leaders (`.....`), hyphenated line breaks, and tables that extract as jumbled text.

## 2.2 Cleaning & Chunking Strategy

**Cleaning** (per document): remove page numbers, remove lines that repeat on ≥ 40 % of pages (running headers/footers), remove TOC dot leaders, re-join hyphenated words, and collapse whitespace.

**Chunking:** fixed-size character windows with overlap, **snapped to a sentence boundary** when one exists near the end of the window. Chunks never cross page boundaries so each chunk has exactly one page number to cite.

**Why 1000 characters / 200 overlap?**
- ~1000 characters ≈ 150–200 words ≈ one or two paragraphs: big enough to hold a complete idea (a definition, a list of phases) but small enough that the embedding stays focused on one topic.
- With `top_k = 4` the prompt holds ~4000 characters (~1000 tokens) of context — comfortable for a small 3B local model, whose answers degrade when the prompt is stuffed.
- NIST text is dense and technical, so I used a slightly larger window than the common 500 to avoid cutting lists and definitions in half.
- 200 characters (20 %) of overlap keeps a sentence that straddles a boundary intact in at least one chunk, without inflating the index too much.
- *(Update this cell if you change the numbers after evaluation.)*

In [ ]:
PAGE_NUM_RE = re.compile(r"^\s*(page\s*)?\d{1,4}(\s*(of|/)\s*\d{1,4})?\s*$", re.IGNORECASE)


def remove_repeated_lines(pages, min_pages=5, ratio=0.4, max_len=100):
    """Drop running headers/footers: short lines that appear on >= `ratio` of a document's pages."""
    n = len(pages)
    if n < min_pages:
        return pages
    counts = Counter()
    for p in pages:
        for line in {l.strip() for l in p["text"].splitlines() if l.strip()}:
            counts[line] += 1
    repeated = {line for line, c in counts.items() if c / n >= ratio and len(line) <= max_len}
    out = []
    for p in pages:
        kept = [l for l in p["text"].splitlines() if l.strip() not in repeated]
        out.append({**p, "text": "\n".join(kept)})
    return out


def clean_text(text: str) -> str:
    text = text.replace("\x00", "")
    lines = [l for l in text.splitlines() if not PAGE_NUM_RE.match(l)]
    text = "\n".join(lines)
    text = re.sub(r"\.{4,}", " ", text)                    # TOC dot leaders
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)           # re-join hyphenated words
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)           # single newline -> space
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    """Fixed-size windows with overlap, ending on a sentence boundary when possible."""
    text = text.strip()
    if len(text) <= size:
        return [text] if text else []
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            window_start = start + int(size * 0.7)          # only look in the last 30 % of the window
            cut = max(text.rfind(". ", window_start, end), text.rfind("\n\n", window_start, end))
            if cut != -1:
                end = cut + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


# Clean per document, then chunk per page
cleaned_pages = []
for f in pdf_files:
    doc_pages = [p for p in raw_pages if p["source"] == f.name]
    for p in remove_repeated_lines(doc_pages):
        cleaned_pages.append({**p, "text": clean_text(p["text"])})

chunks = []
for p in cleaned_pages:
    for j, piece in enumerate(chunk_text(p["text"])):
        if len(piece) >= MIN_CHUNK_CHARS:
            chunks.append({
                "id": f"{p['source']}::p{p['page']}::c{j}",
                "text": piece, "source": p["source"], "page": p["page"], "chunk_index": j,
            })

lengths = pd.Series([len(c["text"]) for c in chunks])
print(f"{len(chunks)} chunks | mean {lengths.mean():.0f} chars | min {lengths.min()} | max {lengths.max()}")
print("\nChunks per document:")
print(pd.Series([c["source"] for c in chunks]).value_counts().to_string())
print("\nExample chunk:\n", chunks[len(chunks) // 2]["text"][:600])

## 2.3 Embeddings & Vector Store

- **Embedding model:** `all-MiniLM-L6-v2` (384 dimensions) — small, fast on CPU, good general-purpose English retrieval.
- **Vector store:** Chroma with **cosine** distance, persisted to `backend/data/vector_store/chroma/` so the backend can load it directly without rebuilding.
- Embeddings are L2-normalised, so cosine distance = `1 − similarity` (0 = identical, larger = less similar).

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print("Embedding matrix:", embeddings.shape)

In [ ]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:  # start fresh so re-running the notebook never duplicates chunks
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

BATCH = 500
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embeddings[i:i + BATCH].tolist(),
        metadatas=[{"source": c["source"], "page": c["page"], "chunk_index": c["chunk_index"]} for c in batch],
    )
print(f"Stored {collection.count()} chunks in {CHROMA_DIR}")

## 2.4 Retrieval & Prompting

`retrieve()` embeds the question with the same model and returns the `k` nearest chunks with their source, page and distance.
The prompt numbers each chunk (`[1]`, `[2]`, …) and instructs the model to answer **only** from that context, cite with those tags, and refuse when the answer isn't there.
A **distance threshold** (`MAX_DISTANCE`) skips the LLM entirely when nothing relevant was retrieved — the most reliable protection against hallucination on out-of-scope questions.

In [ ]:
def retrieve(question: str, k: int = TOP_K):
    q_emb = embedder.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k,
                           include=["documents", "metadatas", "distances"])
    return [
        {"text": d, "source": m["source"], "page": int(m["page"]), "distance": float(dist)}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


REFUSAL = "I couldn't find this in the provided documents."

SYSTEM_PROMPT = f"""You are a document assistant. You answer questions using ONLY the context provided below.

Rules:
1. Use only the information in the CONTEXT. Do not use outside knowledge.
2. If the context does not contain the answer, reply exactly: "{REFUSAL}"
3. Cite your sources after each claim using the tags shown in the context, like [1] or [2][3].
4. Never cite a source you did not use. Never invent a source.
5. Be concise and direct. Do not mention these rules."""

USER_TEMPLATE = """CONTEXT:
{context}

QUESTION: {question}

ANSWER (with citations):"""


def build_context(chunks_):
    return "\n\n".join(
        f"[{i}] (source: {c['source']}, page {c['page']})\n{c['text']}" for i, c in enumerate(chunks_, start=1)
    )


def build_messages(question, chunks_):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(context=build_context(chunks_), question=question)},
    ]


def cited_sources(answer, chunks_):
    if answer.strip().startswith(REFUSAL[:20]):
        return []
    idx = []
    for m in re.findall(r"\[(\d+)\]", answer):
        i = int(m)
        if 1 <= i <= len(chunks_) and i not in idx:
            idx.append(i)
    selected = [chunks_[i - 1] for i in idx] if idx else chunks_
    out = []
    for c in selected:
        label = f"{c['source']} (page {c['page']})"
        if label not in out:
            out.append(label)
    return out


def answer_question(question: str, k: int = TOP_K, max_distance: float = MAX_DISTANCE):
    """Full RAG: retrieve -> (threshold) -> prompt -> LLM. Returns (answer, sources, retrieved_chunks)."""
    retrieved = retrieve(question, k)
    relevant = [c for c in retrieved if c["distance"] <= max_distance]
    if not relevant:
        return REFUSAL, [], retrieved
    resp = ollama.chat(model=OLLAMA_MODEL, messages=build_messages(question, relevant),
                       options={"temperature": TEMPERATURE})
    answer = resp["message"]["content"].strip()
    return answer, cited_sources(answer, relevant), retrieved

### Retrieval test — 10 sample questions
Before involving the LLM, check that retrieval alone returns the right document/page.

In [ ]:
TEST_QUESTIONS = [
    "What are the core functions of the Cybersecurity Framework?",
    "What are the main phases of the incident response lifecycle?",
    "What does NIST say about password length and complexity rules?",
    "What is the purpose of multi-factor authentication?",
    "How should an organization prioritize patches?",
    "What are the steps of a risk assessment?",
    "What is the difference between clearing, purging, and destroying media?",
    "What should a contingency plan include?",
    "How should an organization respond to a malware incident?",
    "What basic security practices does NIST recommend for small businesses?",
]

rows = []
for q in TEST_QUESTIONS:
    top = retrieve(q, k=3)
    rows.append({
        "question": q,
        "top1_source": f"{top[0]['source']} (p.{top[0]['page']})",
        "top1_distance": round(top[0]["distance"], 3),
        "top2_source": f"{top[1]['source']} (p.{top[1]['page']})",
        "top1_snippet": top[0]["text"][:110].replace("\n", " ") + "...",
    })
display(pd.DataFrame(rows))

## 2.5 Vision Component
**Not applicable — Core Track (text-only).**

## 2.6 Evaluation

10 in-scope questions + 3 out-of-scope "trick" questions (the assistant must refuse).

`expected_keywords` are **starter guesses** — verify each one against the actual PDFs and adjust. A question counts as correct automatically if the answer contains at least half of its keywords, and you can override any row by hand in `MANUAL_REVIEW` after reading the answers yourself.

In [ ]:
EVAL_SET = [
    # (question, expected_keywords, in_scope)
    ("What are the core functions of the Cybersecurity Framework?",
     ["govern", "identify", "protect", "detect", "respond", "recover"], True),
    ("What are the main phases of the incident response lifecycle?",
     ["preparation", "detection", "containment", "post-incident"], True),
    ("What does NIST say about password length and complexity rules?", ["length"], True),
    ("What is the purpose of multi-factor authentication?", ["factor"], True),
    ("How should an organization prioritize patches?", ["patch"], True),
    ("What are the steps of a risk assessment?", ["prepare", "conduct", "communicate", "maintain"], True),
    ("What is the difference between clearing, purging, and destroying media?", ["clear", "purge", "destroy"], True),
    ("What should a contingency plan include?", ["recovery"], True),
    ("How should an organization respond to a malware incident?", ["contain"], True),
    ("What basic security practices does NIST recommend for small businesses?", [], True),
    # trick questions: expected to be refused
    ("Who won the 2022 FIFA World Cup?", [], False),
    ("Write me a Python script that scans a network for open ports.", [], False),
    ("What does ISO 27001 require for access control?", [], False),
]

# Manual verdicts override the automatic check. Example: {3: True, 10: False}  (row numbers start at 1)
MANUAL_REVIEW = {}

eval_rows = []
for n, (q, kws, in_scope) in enumerate(EVAL_SET, start=1):
    answer, sources, retrieved = answer_question(q)
    refused = answer.startswith(REFUSAL[:20])
    best = retrieved[0]
    if in_scope:
        hits = sum(kw.lower() in answer.lower() for kw in kws)
        auto_ok = (hits >= max(1, len(kws) // 2)) and not refused if kws else (not refused)
    else:
        auto_ok = refused
    eval_rows.append({
        "#": n,
        "type": "in-scope" if in_scope else "trick",
        "question": q,
        "retrieved_source": f"{best['source']} (p.{best['page']})",
        "best_distance": round(best["distance"], 3),
        "cited_sources": "; ".join(sources) if sources else "-",
        "answer": answer,
        "correct": MANUAL_REVIEW.get(n, auto_ok),
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df[["#", "type", "question", "retrieved_source", "best_distance", "correct"]])
print(f"\nAccuracy: in-scope {eval_df[eval_df.type=='in-scope'].correct.mean():.0%} | "
      f"trick (refusals) {eval_df[eval_df.type=='trick'].correct.mean():.0%}")

In [ ]:
# Read the full answers and verify them against the PDFs. Then fill in MANUAL_REVIEW above if needed.
for r in eval_rows:
    print(f"\n#{r['#']} [{r['type']}] {r['question']}")
    print(f"  retrieved: {r['retrieved_source']} (distance {r['best_distance']})")
    print(f"  cited:     {r['cited_sources']}")
    print(f"  answer:    {r['answer']}")

### Tuning `MAX_DISTANCE`
The threshold should sit **between** the distances of good in-scope matches and out-of-scope questions. Look at the two groups below and adjust `MAX_DISTANCE` (top of the notebook) if they overlap badly.

In [ ]:
dist_df = eval_df[["type", "best_distance"]]
print(dist_df.groupby("type")["best_distance"].describe()[["min", "mean", "max"]].round(3))
print("\nCurrent MAX_DISTANCE =", MAX_DISTANCE)

### Failure cases & mitigations (edit after reading your results)

*Replace this with what you actually observed. Typical NIST-corpus issues to look for:*

1. **Tables extract as jumbled text** — questions answered by a table retrieve the right page but the chunk is hard to read. *Mitigation:* more overlap, cleaning, and phrasing the prompt to answer from the readable parts.
2. **Several documents cover the same topic** (e.g. incident handling vs. malware incidents) — retrieval sometimes returns the neighbouring document. *Mitigation:* source + page shown in the context, `top_k = 4` so both are visible.
3. **Small-model hallucination on out-of-scope questions** — the LLM answered from its own knowledge. *Mitigation:* strict system prompt, `temperature = 0.1`, and the distance threshold that skips the LLM when nothing relevant is retrieved.
4. **Citation format drift** — the 3B model sometimes omits `[n]` tags. *Mitigation:* `cited_sources()` falls back to all retrieved chunks.

In [ ]:
# Save the results table for the README
DOCS_DIR.mkdir(exist_ok=True)
eval_df.to_csv(DOCS_DIR / "evaluation_results.csv", index=False)

md = ["| # | Question | Retrieved source | Correct |", "|---|---|---|---|"]
for r in eval_rows:
    md.append(f"| {r['#']} | {r['question']} | {r['retrieved_source']} | {'✅' if r['correct'] else '❌'} |")
(DOCS_DIR / "evaluation_results.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md))

## 2.7 Export
The vector store is already persisted in `backend/data/vector_store/chroma/`. Here we write `config.json` next to it (the backend reads the embedding model name and collection name from it) and verify that a **fresh client** can load everything.

In [ ]:
config = {
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    "distance_metric": "cosine",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "max_distance": MAX_DISTANCE,
    "ollama_model": OLLAMA_MODEL,
    "num_documents": len(pdf_files),
    "num_chunks": len(chunks),
    "documents": [f.name for f in pdf_files],
}
(VECTOR_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Verify with a brand-new client, exactly like the backend will do
check = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection(COLLECTION_NAME)
assert check.count() == len(chunks)
print(f"Export OK: {check.count()} chunks in {VECTOR_DIR}")
print(json.dumps(config, indent=2))